# 07 — Metrics + diagnostics

Every metric in the project, derived and implemented.

## Roadmap
1. Confusion matrix (the foundation of every classification metric)
2. Precision, recall, F1
3. ROC and PR curves, AUC, AP
4. mAP — what detection papers actually report
5. RMSE, MAE, MAPE, R² (regression)
6. **When to use which** — a decision table you can keep


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Confusion matrix

For one class:

|  | predicted positive | predicted negative |
|---|---|---|
| true positive | **TP** | FN |
| true negative | FP | TN |


In [ ]:
import numpy as np
y_true = np.array([1,1,1,1,0,0,0,0,0,0])
y_pred = np.array([1,1,1,0,1,0,0,0,0,1])

TP = ((y_pred==1) & (y_true==1)).sum()
FP = ((y_pred==1) & (y_true==0)).sum()
FN = ((y_pred==0) & (y_true==1)).sum()
TN = ((y_pred==0) & (y_true==0)).sum()
print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}")


## 2. Precision, Recall, F1

$$\text{precision} = \frac{TP}{TP + FP}, \quad \text{recall} = \frac{TP}{TP + FN}$$

$$F_1 = \frac{2 P R}{P + R}$$

Precision: "when I said yes, was I right?"  Recall: "of all the yeses, how many did I catch?"


In [ ]:
P = TP / max(TP + FP, 1)
R = TP / max(TP + FN, 1)
F1 = 2 * P * R / max(P + R, 1e-9)
print(f"precision={P:.3f}  recall={R:.3f}  F1={F1:.3f}")


## 3. Precision–recall curve and Average Precision

The model usually outputs a probability, not a hard yes/no. The threshold for "yes" affects precision and recall. **AP** is the area under the precision–recall curve as you sweep the threshold.


In [ ]:
import matplotlib.pyplot as plt

# 5 predictions with their probabilities, sorted by descending prob
probs  = np.array([0.95, 0.80, 0.65, 0.50, 0.20])
labels = np.array([1,    1,    0,    1,    0])      # ground truth

# Walk the threshold from high to low
prec_list, rec_list = [], []
tp = fp = 0
N = labels.sum()
for p, lab in sorted(zip(probs, labels), key=lambda kv: -kv[0]):
    if lab == 1: tp += 1
    else:        fp += 1
    prec_list.append(tp / (tp + fp))
    rec_list.append(tp / N)

# Trapezoidal AUC of PR curve = AP
AP = float(np.trapz(prec_list, rec_list))
plt.figure(figsize=(5, 4))
plt.plot(rec_list, prec_list, "o-"); plt.xlabel("recall"); plt.ylabel("precision")
plt.title(f"AP ≈ {AP:.3f}"); plt.xlim(0,1); plt.ylim(0,1.05); plt.grid(True); plt.show()


**mAP** = mean of AP across all classes. `mAP@0.5` means with IoU threshold 0.5 for matching predicted boxes to ground truth.

## 4. Regression metrics


In [ ]:
y_true = np.array([100, 200, 300, 400], dtype=float)
y_pred = np.array([110, 180, 330, 360], dtype=float)
err = y_pred - y_true

RMSE = np.sqrt((err**2).mean())
MAE  = np.abs(err).mean()
MAPE = (np.abs(err) / np.abs(y_true)).mean() * 100

ss_res = (err**2).sum()
ss_tot = ((y_true - y_true.mean())**2).sum()
R2 = 1 - ss_res / ss_tot

print(f"RMSE = {RMSE:.2f}")
print(f"MAE  = {MAE:.2f}")
print(f"MAPE = {MAPE:.2f} %")
print(f"R²   = {R2:.4f}")


## 5. Decision table — which metric should I report?

| Situation | Metric |
|---|---|
| Balanced 2-class classification | accuracy is fine |
| Imbalanced 2-class (95% negative) | **F1** or **PR-AUC** — accuracy will look great even for a useless model |
| Multi-class | macro-F1 (average across classes) |
| Multi-label | per-class F1 + macro-F1 |
| Object detection | **mAP@0.5** and **mAP@0.5:0.95** |
| Regression for end-user dollars | **MAE** + **MAPE** (interpretable) |
| Regression where outliers matter | **RMSE** |
| Anything safety-critical | recall, specifically the false-negative rate |
